In [1]:
# CELL 1 — COMMON INFRASTRUCTURE, DATASET ADAPTER, METRICS, TRAINING UTILITIES
# ============================================================================
# Task: Image ↔ Caption Retrieval Failure Detection (Class-less setting)
# ============================================================================

import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
import os, re, json, math, copy, random, warnings, subprocess, sys, gc, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Callable, Optional, Tuple, Any, Union

from scipy.special import softmax as scipy_softmax
from IPython.display import display

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

from torchvision import transforms
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.model_selection import train_test_split

try:
    import clip
except ImportError as exc:
    raise ImportError(
        "Install OpenAI CLIP first:\n"
        "pip install git+https://github.com/openai/CLIP.git"
    ) from exc

# ============================================================================
# USER CONFIG — Image-Caption Retrieval Failure Detection
# ============================================================================

@dataclass
class Config:
    # ─── Dataset identification ─────────────────────────────────────────
    dataset_name: str = "Dataset"
    data_root: str = "/./dataset"
    results_dir: str = "/./Dataset/failure_detection_results"

    # ─── Split / annotation source ───────────────────────────────────────
    splits_dir: str = "dataset_datasplit_all"
    train_tsv: str = "task_informative_text_img_train.tsv"
    val_tsv: str = "task_informative_text_img_dev.tsv"
    test_tsv: str = "task_informative_text_img_test.tsv"
    image_root_subdir: str = "."  
    min_caption_chars: int = 3   

    # ─── Data split configuration ───────────────────────────────────────
    # NOTE: Change these three values to modify the train/val/test partition.
    #        They must sum to 1.0. Example: (0.7, 0.15, 0.15).
    split_ratio: Tuple[float, float, float] = (0.925, 0.05, 0.025)
    split_tag: str = "split_92.5_5_2.5"
    split_seed: int = 42

    # ─── Retrieval / candidate pool settings ────────────────────────────
    candidate_pool_mode: str = "split"
    zero_shot_target_models: Tuple[str, ...] = ("CLIP", "SigLIP", "OpenCLIP")

    # ─── Target VLM zoo ─────────────────────────────────────────────────
    clip_model: str = "ViT-B/32"
    logit_scale: float = 100.0

    # ─── Feature extraction ─────────────────────────────────────────────
    zeroshot_batch_size: int = 32
    clear_cuda_each_zeroshot_batch: bool = True
    feature_batch_size: int = 256
    num_workers: int = 4
    use_amp: bool = True

    # ─── Training / evaluation ──────────────────────────────────────────
    seeds: Tuple[int, ...] = (42, 43, 44, 45, 46)
    max_epochs: int = 100
    patience: int = 15
    lr_grid: Tuple[float, ...] = (1e-3, 1e-2, 1e-1)
    batch_size_grid: Tuple[int, ...] = (256, 512)
    n_iter_freeze_proj: int = 1000
    caption_aux_weight: float = 0.10


CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = bool(CFG.use_amp and torch.cuda.is_available())
ACTIVE_FEATURE_DIM = 512
ACTIVE_TEXT_DIM = 512


def set_active_dims(encoded=None, text_features=None):
    global ACTIVE_FEATURE_DIM, ACTIVE_TEXT_DIM
    if encoded is not None:
        ACTIVE_FEATURE_DIM = int(encoded["train"]["v"].shape[1])
    if text_features is not None:
        ACTIVE_TEXT_DIM = int(text_features.shape[1])


RESULTS_DIR = Path(CFG.results_dir)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================================
# REPRODUCIBILITY
# ============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clean_device_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def _np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


# ============================================================================
# IMAGE PREPROCESSING (CLIP-style)
# ============================================================================

class ConvertImageToRGB:
    def __call__(self, image):
        return image.convert("RGB") if isinstance(image, Image.Image) else image


def clip_style_transform():
    return transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        ConvertImageToRGB(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.48145466, 0.4578275, 0.40821073],
            std=[0.26862954, 0.26130258, 0.27577711],
        ),
    ])


val_transforms = clip_style_transform()


# ============================================================================
# GENERIC TEXT CLEANING
# ============================================================================

_URL_RE = re.compile(r"http\S+|www\.\S+")
_MENTION_RE = re.compile(r"@\w+")
_HASHTAG_RE = re.compile(r"#(\w+)")
_WS_RE = re.compile(r"\s+")
_RT_RE = re.compile(r"^RT\s+", flags=re.IGNORECASE)


def clean_caption(text: str) -> str:
    """Generic text cleaner. Strips URLs, mentions, hashtag symbols, RT prefix, HTML entities."""
    if text is None:
        return ""
    s = str(text)
    s = _RT_RE.sub("", s)
    s = _URL_RE.sub(" ", s)
    s = _MENTION_RE.sub(" ", s)
    s = _HASHTAG_RE.sub(r"\1", s)
    s = s.replace("&amp;", "&").replace("&gt;", ">").replace("&lt;", "<")
    s = _WS_RE.sub(" ", s).strip()
    return s


# Backwards-compatible alias for dataset adapter
clean_tweet = clean_caption


# ============================================================================
# IMAGE-CAPTION PAIR DATASET
# ============================================================================

class ImageCaptionPairDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]], transform=None):
        self.pairs = list(pairs)
        self.transform = transform
        self.targets = list(range(len(self.pairs)))
        self.captions = [c for _, c in self.pairs]
        self.image_paths = [p for p, _ in self.pairs]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        path, caption = self.pairs[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return {
            "image": image,
            "caption": caption,
            "target": int(index),
            "index": int(index),
            "path": str(path),
        }


# ============================================================================
# DATASET ADAPTER (Encapsulated)
# ============================================================================

def _read_dataset_tsv(tsv_path: Path) -> pd.DataFrame:
    if not tsv_path.exists():
        raise FileNotFoundError(f"Missing dataset split file: {tsv_path}")
    df = pd.read_csv(tsv_path, sep="\t", dtype=str, keep_default_na=False,
                     na_values=[""], encoding="utf-8", on_bad_lines="skip")
    df.columns = [c.strip() for c in df.columns]
    return df


def _resolve_image_path(data_root: Path, rel_or_abs: str) -> Optional[Path]:
    if rel_or_abs is None or not str(rel_or_abs).strip():
        return None
    p = str(rel_or_abs).strip()
    candidates = [
        Path(p),
        data_root / p,
        data_root / "data_image" / p,
    ]
    for c in candidates:
        if c.exists():
            return c
    return None


def _extract_pairs_from_dataset_tsv(df: pd.DataFrame, data_root: Path, min_chars: int
                                       ) -> List[Tuple[str, str]]:
    text_col = None
    for cand in ("tweet_text", "text", "tweet"):
        if cand in df.columns:
            text_col = cand
            break
    if text_col is None:
        raise KeyError("No text column found in Dataset TSV.")

    img_col = None
    for cand in ("image", "image_path", "image_id"):
        if cand in df.columns:
            img_col = cand
            break
    if img_col is None:
        raise KeyError("No image column found in Dataset TSV.")

    pairs, seen = [], set()
    for _, row in df.iterrows():
        raw_path = row.get(img_col, "")
        raw_text = row.get(text_col, "")
        resolved = _resolve_image_path(data_root, raw_path)
        if resolved is None:
            continue
        key = str(resolved)
        if key in seen:
            continue
        caption = clean_caption(raw_text)
        if len(caption) < int(min_chars):
            continue
        seen.add(key)
        pairs.append((key, caption))
    return pairs


def dataset_pairs_provider(cfg: Config) -> List[Tuple[str, str]]:
    root = Path(cfg.data_root).expanduser()
    if not root.exists():
        raise FileNotFoundError(f"Dataset data_root not found: {root}")

    split_dir = root / cfg.splits_dir
    if not split_dir.exists():
        alt = root / "dataset_datasplit_with_LLaVA_generated_text"
        if alt.exists():
            split_dir = alt
        else:
            raise FileNotFoundError(f"Missing Dataset splits dir: {split_dir}")

    all_pairs, seen_paths = [], set()
    for tsv_name in (cfg.train_tsv, cfg.val_tsv, cfg.test_tsv):
        tsv_path = split_dir / tsv_name
        if tsv_path.exists():
            df = _read_dataset_tsv(tsv_path)
            pairs = _extract_pairs_from_dataset_tsv(df, root, cfg.min_caption_chars)
            for path, caption in pairs:
                if path not in seen_paths:
                    seen_paths.add(path)
                    all_pairs.append((path, caption))
    return all_pairs


# ============================================================================
# GENERIC DATASET BUNDLE BUILDER (Dataset- and Split-Agnostic)
# ============================================================================

def _filter_and_dedupe_pairs(pairs: List[Tuple[str, str]],
                             min_chars: int,
                             clean_fn: Optional[Callable[[str], str]] = None
                             ) -> List[Tuple[str, str]]:
    if clean_fn is None:
        clean_fn = clean_caption
    out, seen = [], set()
    for path, caption in pairs:
        if path is None:
            continue
        path_str = str(path)
        if path_str in seen:
            continue
        if not Path(path_str).exists():
            continue
        cap = clean_fn(caption)
        if len(cap) < int(min_chars):
            continue
        seen.add(path_str)
        out.append((path_str, cap))
    return out


def build_dataset_bundle(
    cfg: Config,
    pairs_provider: Union[
        List[Tuple[str, str]],
        Dict[str, List[Tuple[str, str]]],
        Callable[[Config], Union[List[Tuple[str, str]], Dict[str, List[Tuple[str, str]]]]],
    ] = dataset_pairs_provider,
    clean_fn: Optional[Callable[[str], str]] = None,
    transform=None,
) -> Dict[str, Any]:
    """Builds train/val/test datasets. Split ratio and seed come from CFG."""
    if transform is None:
        transform = val_transforms

    split_ratio = cfg.split_ratio
    split_seed = cfg.split_seed

    # Resolve provider
    if callable(pairs_provider):
        provided = pairs_provider(cfg)
    else:
        provided = pairs_provider

    # Case B: pre-split dict
    if isinstance(provided, dict):
        required = {"train", "val", "test"}
        if not required.issubset(provided.keys()):
            raise ValueError(
                f"pairs_provider dict must contain keys {required}, got {set(provided.keys())}"
            )
        train_pairs = _filter_and_dedupe_pairs(provided["train"], cfg.min_caption_chars, clean_fn)
        val_pairs = _filter_and_dedupe_pairs(provided["val"], cfg.min_caption_chars, clean_fn)
        test_pairs = _filter_and_dedupe_pairs(provided["test"], cfg.min_caption_chars, clean_fn)

    # Case A: flat pool → split by ratio
    elif isinstance(provided, (list, tuple)):
        all_pairs = _filter_and_dedupe_pairs(list(provided), cfg.min_caption_chars, clean_fn)
        all_pairs = sorted(all_pairs, key=lambda x: x[0])
        rng = np.random.default_rng(int(split_seed))
        rng.shuffle(all_pairs)

        total = len(all_pairs)
        train_frac, val_frac, test_frac = split_ratio
        if not math.isclose(train_frac + val_frac + test_frac, 1.0, abs_tol=1e-6):
            raise ValueError(f"cfg.split_ratio must sum to 1.0, got {split_ratio}")

        n_train = int(round(total * train_frac))
        n_val = int(round(total * val_frac))

        train_pairs = all_pairs[:n_train]
        val_pairs = all_pairs[n_train:n_train + n_val]
        test_pairs = all_pairs[n_train + n_val:]
    else:
        raise TypeError(
            f"pairs_provider must be list, dict, or callable; got {type(provided)}"
        )

    train_ds = ImageCaptionPairDataset(train_pairs, transform=transform)
    val_ds = ImageCaptionPairDataset(val_pairs, transform=transform)
    test_ds = ImageCaptionPairDataset(test_pairs, transform=transform)

    captions = {
        "train": train_ds.captions,
        "val": val_ds.captions,
        "test": test_ds.captions,
    }

    train_frac, val_frac, test_frac = split_ratio
    print(f"[{cfg.dataset_name}] Split sizes [{train_frac*100:g}/{val_frac*100:g}/{test_frac*100:g}]: "
          f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

    return {
        "train": train_ds,
        "val": val_ds,
        "test": test_ds,
        "captions": captions,
        "class_names": captions["train"],
        "is_classless": True,
        "task": "image_caption_retrieval_failure",
    }


# ============================================================================
# METRICS & EVALUATION
# ============================================================================

def get_auroc(success_scores, failure_scores):
    y = np.r_[np.zeros(len(success_scores)), np.ones(len(failure_scores))]
    s = np.r_[success_scores, failure_scores]
    return float(roc_auc_score(y, s))


def get_fpr95(success_scores, failure_scores):
    y = np.r_[np.zeros(len(success_scores)), np.ones(len(failure_scores))]
    s = np.r_[success_scores, failure_scores]
    fpr, tpr, _ = roc_curve(y, s)
    idx95 = np.where(tpr >= 0.95)[0]
    return float(fpr[idx95[0]]) if len(idx95) > 0 else 1.0


def normalize_uncertainty_scores(scores):
    scores = np.asarray(scores, dtype=float).reshape(-1)
    lo, hi = np.nanmin(scores), np.nanmax(scores)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(scores, dtype=float)
    return (scores - lo) / (hi - lo)


def cohen_d_between(success_scores, failure_scores):
    s = np.asarray(success_scores, dtype=float).reshape(-1)
    f = np.asarray(failure_scores, dtype=float).reshape(-1)
    if len(s) < 2 or len(f) < 2:
        return float("nan")
    a = normalize_uncertainty_scores(np.r_[s, f])
    su, fu = a[:len(s)], a[len(s):]
    pooled = math.sqrt((np.std(fu) ** 2 + np.std(su) ** 2) / 2.0) + 1e-12
    return float((np.mean(fu) - np.mean(su)) / pooled)


def youden_threshold_for_uncertainty(scores, correct_mask):
    u = normalize_uncertainty_scores(scores)
    fl = (~np.asarray(correct_mask, dtype=bool)).astype(int)
    if len(np.unique(fl)) < 2:
        return float("nan")
    fpr, tpr, th = roc_curve(fl, u)
    return float(th[int(np.nanargmax(tpr - fpr))])


def evaluate_failure_scores(scores, labels, image_features, text_features,
                            logit_scale=100.0, correct_mask=None,
                            target_name="CLIP"):
    scores = np.asarray(scores, dtype=float).reshape(-1)
    
    img_f = torch.as_tensor(image_features, dtype=torch.float32, device=DEVICE)
    txt_f = torch.as_tensor(text_features, dtype=torch.float32, device=DEVICE)
    
    if correct_mask is None:
        with torch.no_grad():
            sims = logit_scale * img_f @ txt_f.T
            preds = sims.argmax(dim=-1)
            correct = labels.to(DEVICE) == preds
        correct_np = correct.cpu().numpy().astype(bool)
    else:
        correct_np = np.asarray(correct_mask, dtype=bool).reshape(-1)
        if len(correct_np) != len(scores):
            raise ValueError(f"correct_mask length mismatch")

    success, failure = scores[correct_np], scores[~correct_np]
    if len(success) == 0 or len(failure) == 0:
        auc = fpr95 = cohen_d = youden_theta = float("nan")
    else:
        auc = get_auroc(success, failure)
        fpr95 = get_fpr95(success, failure)
        cohen_d = cohen_d_between(success, failure)
        youden_theta = youden_threshold_for_uncertainty(scores, correct_np)

    return {
        "target_model": str(target_name),
        "accuracy": float(np.mean(correct_np)),
        "auc": auc,
        "fpr95": fpr95,
        "cohen_d": cohen_d,
        "youden_theta": youden_theta,
        "n_success": int(np.sum(correct_np)),
        "n_failure": int(np.sum(~correct_np)),
        "scores": scores,
        "correct": correct_np,
    }


# ============================================================================
# ROBUST LOSS ARCHITECTURES
# ============================================================================

class FailureBCELoss(nn.Module):
    def forward(self, logits, probs, labels, failure_targets=None):
        if failure_targets is not None:
            failure = failure_targets.float()
        else:
            failure = (labels != probs.argmax(dim=-1)).float()
        n_fail = failure.sum().clamp_min(1.0)
        n_success = (1.0 - failure).sum().clamp_min(1.0)
        pw = (n_success / n_fail).clamp(max=30.0).detach()
        return F.binary_cross_entropy_with_logits(
            logits, failure, pos_weight=pw, reduction="mean"
        )


class FixedWeightFailureBCELoss(nn.Module):
    def __init__(self, pos_weight: float = 1.0):
        super().__init__()
        self.register_buffer("pos_weight", torch.tensor([pos_weight]))

    def forward(self, logits, probs, labels, failure_targets=None):
        if failure_targets is not None:
            failure = failure_targets.float()
        else:
            failure = (labels != probs.argmax(dim=-1)).float()
        return F.binary_cross_entropy_with_logits(
            logits, failure, pos_weight=self.pos_weight.to(logits.device), reduction="mean"
        )


class OrdinaryBCELoss(nn.Module):
    def forward(self, logits, probs, labels, failure_targets=None):
        if failure_targets is not None:
            failure = failure_targets.float()
        else:
            failure = (labels != probs.argmax(dim=-1)).float()
        return F.binary_cross_entropy_with_logits(logits, failure, reduction="mean")


class ViLUWeightedBCELoss(nn.Module):
    """Log-weighted BCE for ViLU-style retrieval-failure heads (paper Eq. 9-10)."""
    def forward(self, logits, probs, labels, failure_targets=None):
        if failure_targets is not None:
            failure = failure_targets.float()
        else:
            failure = (labels != probs.argmax(dim=-1)).float()
        n_fail = failure.sum().clamp_min(1.0)
        n_success = (1.0 - failure).sum().clamp_min(1.0)
        pw = torch.log1p(n_success / n_fail).detach()
        w = torch.where(failure > 0.5, pw, torch.ones_like(failure))
        return F.binary_cross_entropy_with_logits(logits, failure, weight=w, reduction="mean")


def target_scores_to_probs(scores, activation="softmax"):
    return torch.sigmoid(scores) if str(activation).lower() == "sigmoid" \
        else torch.softmax(scores, dim=-1)


# ============================================================================
# PAIRWISE RANKING OBJECTIVE
# ============================================================================

class PairwiseRankingLoss(nn.Module):
    def __init__(self, sample_hard: bool = False, max_pairs: int = 256):
        super().__init__()
        self.sample_hard = sample_hard
        self.max_pairs = max_pairs

    def forward(self, logits: torch.Tensor, failure_targets: torch.Tensor) -> torch.Tensor:
        fails_idx = torch.where(failure_targets > 0.5)[0]
        succs_idx = torch.where(failure_targets <= 0.5)[0]

        if len(fails_idx) == 0 or len(succs_idx) == 0:
            return torch.tensor(0.0, device=logits.device)

        s_f = logits[fails_idx]
        s_s = logits[succs_idx]

        if self.sample_hard:
            s_f_sorted, _ = torch.sort(s_f, descending=False)
            s_s_sorted, _ = torch.sort(s_s, descending=True)
            n_pairs = min(len(s_f_sorted), len(s_s_sorted), self.max_pairs)
            if n_pairs == 0:
                return torch.tensor(0.0, device=logits.device)
            pairs_f = s_f_sorted[:n_pairs]
            pairs_s = s_s_sorted[:n_pairs]
        else:
            n_pairs = min(len(s_f), len(s_s), self.max_pairs)
            if n_pairs == 0:
                return torch.tensor(0.0, device=logits.device)
            f_perm = torch.randperm(len(s_f), device=logits.device)[:n_pairs]
            s_perm = torch.randperm(len(s_s), device=logits.device)[:n_pairs]
            pairs_f = s_f[f_perm]
            pairs_s = s_s[s_perm]

        loss = torch.log1p(torch.exp(-(pairs_f - pairs_s)))
        return loss.mean()


# ============================================================================
# LR / OPTIMIZATION HELPERS
# ============================================================================

def cosine_warmup_lr(base_lr, epoch, total_epochs, warmup=5, min_ratio=0.05):
    if total_epochs <= 0:
        return float(base_lr)
    warmup = max(1, int(warmup))
    if epoch < warmup:
        return float(base_lr) * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total_epochs - warmup)
    return float(base_lr) * (min_ratio + (1.0 - min_ratio) * 0.5 *
                             (1.0 + math.cos(math.pi * progress)))


def set_lr(optimizer, lr):
    for g in optimizer.param_groups:
        g["lr"] = float(lr) * float(g.get("lr_scale", 1.0))


# ============================================================================
# BALANCED SAMPLER
# ============================================================================

class UniversalBalancedSampler:
    def __init__(self, labels, fails, batch_size=128, fail_ratio=0.5,
                 n_batches=None, seed=42):
        if labels is None:
            labels = np.zeros(len(fails), dtype=np.int64)
        self.labels = np.asarray(labels)
        self.fails = np.asarray(fails).astype(int)
        self.batch_size = int(batch_size)
        self.n_fail = max(1, int(round(self.batch_size * float(fail_ratio))))
        self.n_success = max(1, self.batch_size - self.n_fail)
        self.n_batches = int(
            n_batches or max(1, math.ceil(len(self.labels) / self.batch_size))
        )
        self.seed = int(seed)
        self.fail_map, self.success_map = {}, {}
        for c in np.unique(self.labels):
            fi = np.where((self.labels == c) & (self.fails == 1))[0]
            si = np.where((self.labels == c) & (self.fails == 0))[0]
            if len(fi) > 0:
                self.fail_map[int(c)] = fi
            if len(si) > 0:
                self.success_map[int(c)] = si
        self.fail_classes = list(self.fail_map.keys())
        self.success_classes = list(self.success_map.keys())
        self._epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch)
        self._epoch += 1
        for _ in range(self.n_batches):
            batch = []
            if self.fail_classes:
                chosen = rng.choice(self.fail_classes, size=self.n_fail, replace=True)
                batch.extend(int(rng.choice(self.fail_map[int(c)])) for c in chosen)
            if self.success_classes:
                chosen = rng.choice(self.success_classes, size=self.n_success, replace=True)
                batch.extend(int(rng.choice(self.success_map[int(c)])) for c in chosen)
            while len(batch) < self.batch_size:
                batch.append(int(rng.integers(0, len(self.labels))))
            rng.shuffle(batch)
            yield batch[:self.batch_size]

    def __len__(self):
        return self.n_batches


# ============================================================================
# LOADER + SCORING HELPERS (With Dynamic Signature Inspection)
# ============================================================================

def make_loader(v, y, batch_size, seed, shuffle=True, failure_labels=None,
                pred_labels=None, target_logits=None, balanced_sampler_fails=None):
    v = torch.as_tensor(v, dtype=torch.float32)
    
    if balanced_sampler_fails is not None:
        sampler = UniversalBalancedSampler(
            labels=None, fails=balanced_sampler_fails, batch_size=batch_size,
            fail_ratio=0.5, seed=seed
        )
        tensors = [v, y.long()]
        if pred_labels is not None:
            tensors.append(torch.as_tensor(pred_labels).long())
        if failure_labels is not None:
            tensors.append(torch.as_tensor(failure_labels).float())
        if target_logits is not None:
            tensors.append(torch.as_tensor(target_logits).float())
        return DataLoader(
            TensorDataset(*tensors), batch_sampler=sampler, num_workers=0
        )

    g = torch.Generator()
    g.manual_seed(seed)
    tensors = [v, y.long()]
    if pred_labels is not None:
        tensors.append(torch.as_tensor(pred_labels).long())
    if failure_labels is not None:
        tensors.append(torch.as_tensor(failure_labels).float())
    if target_logits is not None:
        tensors.append(torch.as_tensor(target_logits).float())
    return DataLoader(
        TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle,
        generator=g, num_workers=0, drop_last=False,
    )


@torch.no_grad()
def score_torch_model(model, v, t, inference_batch_size=1024,
                      pred_labels=None, true_labels=None):
    model.eval()
    out = []
    bs = max(1, int(inference_batch_size))
    
    t_dev = torch.as_tensor(t, dtype=torch.float32, device=DEVICE)
    
    forward_params = inspect.signature(model.forward).parameters
    
    for start in range(0, len(v), bs):
        sl = slice(start, min(start + bs, len(v)))
        z = torch.as_tensor(v[sl], dtype=torch.float32, device=DEVICE)
        
        kwargs = {}
        if "pred_labels" in forward_params and pred_labels is not None:
            kwargs["pred_labels"] = torch.as_tensor(pred_labels[sl]).long().to(DEVICE)
        if "true_labels" in forward_params and true_labels is not None:
            kwargs["true_labels"] = torch.as_tensor(true_labels[sl]).long().to(DEVICE)
            
        with autocast("cuda", enabled=USE_AMP):
            mo = model(z, t_dev, **kwargs)
            logits = mo[0] if isinstance(mo, tuple) else mo
            out.append(torch.sigmoid(logits.squeeze(-1)).float().cpu())
    return torch.cat(out).numpy()


# ============================================================================
# EXTENDED REGIME NEURAL TRAINING LOOP
# ============================================================================

def train_one_neural_setting(
    model_builder, train_v, train_y, val_v, val_y, text_features, seed,
    lr, batch_size, verbose=False, loss_builder=None,
    train_failure_labels=None, train_pred_labels=None,
    val_pred_labels=None, val_correct_mask=None,
    target_name="CLIP", train_target_logits=None, val_target_logits=None,
    fixed_pos_weight: Optional[float] = None,
    balanced_sampling: bool = False,
    ranking_weight: float = 0.0,
    ranking_type: str = "none",
    atlas_scheduled_aux: bool = False,
):
    set_seed(seed)
    model = model_builder().to(DEVICE)

    if loss_builder is not None:
        loss_fn = loss_builder()
    elif fixed_pos_weight is not None:
        loss_fn = FixedWeightFailureBCELoss(pos_weight=fixed_pos_weight)
    elif balanced_sampling:
        loss_fn = OrdinaryBCELoss()
    else:
        loss_fn = FailureBCELoss()

    if ranking_type == "hard":
        rank_loss_fn = PairwiseRankingLoss(sample_hard=True, max_pairs=batch_size)
    elif ranking_type == "random":
        rank_loss_fn = PairwiseRankingLoss(sample_hard=False, max_pairs=batch_size)
    else:
        rank_loss_fn = None

    optimizer = optim.SGD(model.parameters(), lr=float(lr))

    loader = make_loader(
        train_v, train_y, int(batch_size), seed, shuffle=not balanced_sampling,
        failure_labels=train_failure_labels, pred_labels=train_pred_labels,
        target_logits=train_target_logits,
        balanced_sampler_fails=train_failure_labels if balanced_sampling else None
    )
    
    t = torch.as_tensor(text_features, dtype=torch.float32, device=DEVICE)
    
    forward_params = inspect.signature(model.forward).parameters

    best_state, best_auc, best_fpr, best_epoch, bad = None, -np.inf, np.inf, 0, 0
    history = []

    for epoch in range(CFG.max_epochs):
        model.train()
        losses = []
        for batch in loader:
            cursor = 2
            z_v, labels = batch[0], batch[1]
            pred_labels = batch[cursor].to(DEVICE) if train_pred_labels is not None else None
            if train_pred_labels is not None:
                cursor += 1
            failure_targets = batch[cursor].to(DEVICE) if train_failure_labels is not None else None
            if train_failure_labels is not None:
                cursor += 1
            batch_target_logits = batch[cursor].to(DEVICE) if train_target_logits is not None else None

            z_v = torch.as_tensor(z_v, dtype=torch.float32, device=DEVICE)
            labels = labels.to(DEVICE)
            
            with torch.no_grad():
                if batch_target_logits is None:
                    probs = torch.softmax(CFG.logit_scale * z_v @ t.T, dim=-1)
                else:
                    probs = target_scores_to_probs(batch_target_logits, "softmax")

            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=USE_AMP):
                kwargs = {}
                if "pred_labels" in forward_params and pred_labels is not None:
                    kwargs["pred_labels"] = pred_labels
                
                mo = model(z_v, t, **kwargs)
                logits = mo[0] if isinstance(mo, tuple) else mo
                logits = logits.squeeze(-1)

                cls_loss = loss_fn(logits, probs, labels, failure_targets=failure_targets)
                loss = cls_loss

                if rank_loss_fn is not None and failure_targets is not None:
                    loss = loss + ranking_weight * rank_loss_fn(logits, failure_targets)

                if isinstance(mo, tuple) and len(mo) >= 3:
                    if atlas_scheduled_aux:
                        if epoch < 10:
                            current_aux_weight = 0.10
                        elif epoch < 25:
                            current_aux_weight = 0.05
                        elif epoch < 40:
                            current_aux_weight = 0.01
                        else:
                            current_aux_weight = 0.0
                    else:
                        current_aux_weight = CFG.caption_aux_weight

                    if current_aux_weight > 0:
                        loss = loss + current_aux_weight * (1 - (mo[1] * mo[2]).sum(-1)).mean()

            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        vs = score_torch_model(model, val_v, text_features,
                               inference_batch_size=max(256, batch_size),
                               pred_labels=val_pred_labels)
        vr = evaluate_failure_scores(vs, val_y, val_v, text_features,
                                     CFG.logit_scale,
                                     correct_mask=val_correct_mask,
                                     target_name=target_name)
        auc, fpr = float(vr["auc"]), float(vr["fpr95"])
        history.append({
            "epoch": epoch + 1,
            "loss": float(np.mean(losses)) if losses else float("nan"),
            "val_auc": auc, "val_fpr95": fpr,
        })

        improved = np.isfinite(auc) and (
            auc > best_auc + 1e-12 or
            (abs(auc - best_auc) <= 1e-12 and fpr < best_fpr)
        )
        if improved:
            best_auc, best_fpr, best_epoch = auc, fpr, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1

        if bad >= CFG.patience:
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return model, {
        "lr": float(lr), "batch_size": int(batch_size),
        "best_epoch": int(best_epoch),
        "val_auc": float(best_auc), "val_fpr95": float(best_fpr),
        "history": history,
    }


def grid_search_then_train_final(
    model_builder, train_v, train_y, val_v, val_y, text_features, seed,
    fixed_selection=None, loss_builder=None,
    train_failure_labels=None, train_pred_labels=None,
    val_pred_labels=None, val_correct_mask=None,
    target_name="CLIP", train_target_logits=None, val_target_logits=None,
    fixed_pos_weight: Optional[float] = None,
    balanced_sampling: bool = False,
    ranking_weight: float = 0.0,
    ranking_type: str = "none",
    atlas_scheduled_aux: bool = False,
):
    search_results = []
    if fixed_selection is None:
        best = None
        for lr in CFG.lr_grid:
            for batch_size in CFG.batch_size_grid:
                candidate, info = train_one_neural_setting(
                    model_builder, train_v, train_y, val_v, val_y, text_features,
                    seed=seed, lr=lr, batch_size=batch_size, verbose=False,
                    loss_builder=loss_builder,
                    train_failure_labels=train_failure_labels,
                    train_pred_labels=train_pred_labels,
                    val_pred_labels=val_pred_labels,
                    val_correct_mask=val_correct_mask,
                    target_name=target_name,
                    train_target_logits=train_target_logits,
                    val_target_logits=val_target_logits,
                    fixed_pos_weight=fixed_pos_weight,
                    balanced_sampling=balanced_sampling,
                    ranking_weight=ranking_weight,
                    ranking_type=ranking_type,
                    atlas_scheduled_aux=atlas_scheduled_aux,
                )
                search_results.append(info)
                if best is None:
                    best = info
                else:
                    if info["val_auc"] > best["val_auc"] + 1e-12 or \
                       (abs(info["val_auc"] - best["val_auc"]) <= 1e-12 and
                        info["val_fpr95"] < best["val_fpr95"]):
                        best = info
                del candidate
                clean_device_cache()
    else:
        best = {
            "lr": float(fixed_selection["lr"]),
            "batch_size": int(fixed_selection["batch_size"]),
            "val_auc": float(fixed_selection.get("val_auc", float("nan"))),
            "val_fpr95": float(fixed_selection.get("val_fpr95", float("nan"))),
            "best_epoch": int(fixed_selection.get("best_epoch", 0)),
            "reused_from_grid_seed": fixed_selection.get("grid_seed"),
        }

    final_model, final_info = train_one_neural_setting(
        model_builder, train_v, train_y, val_v, val_y, text_features,
        seed=seed, lr=best["lr"], batch_size=best["batch_size"], verbose=False,
        loss_builder=loss_builder,
        train_failure_labels=train_failure_labels,
        train_pred_labels=train_pred_labels,
        val_pred_labels=val_pred_labels,
        val_correct_mask=val_correct_mask,
        target_name=target_name,
        train_target_logits=train_target_logits,
        val_target_logits=val_target_logits,
        fixed_pos_weight=fixed_pos_weight,
        balanced_sampling=balanced_sampling,
        ranking_weight=ranking_weight,
        ranking_type=ranking_type,
        atlas_scheduled_aux=atlas_scheduled_aux,
    )
    return final_model, {
        "selected": best,
        "final_validation": final_info,
        "grid": search_results,
    }


# ============================================================================
# SAFE CV + SUMMARY HELPERS
# ============================================================================

def _safe_binary_cv(labels, max_cv=3):
    labels = np.asarray(labels)
    unique, counts = np.unique(labels, return_counts=True)
    if len(unique) < 2:
        return 2
    return max(2, min(int(max_cv), int(counts.min())))


def summarize_runs(raw_df):
    numeric = ["auc", "fpr95", "cohen_d"]
    rows = []
    for model, grp in raw_df.groupby("model", sort=False):
        row = {"model": model, "n_runs": len(grp)}
        for col in numeric:
            vals = grp[col].astype(float).to_numpy()
            row[f"{col}_mean"] = float(np.nanmean(vals))
            row[f"{col}_variance"] = float(np.nanvar(vals, ddof=1)) if len(vals) > 1 else 0.0
            row[f"{col}_sd"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

In [2]:
# CELL 2 — ViLU (IMAGE ↔ CAPTION RETRIEVAL FAILURE DETECTION)
# ============================================================================

#
# For query image i:
#   • paired_idx  = i               (ground-truth index in T_pool)
#   • pred_idx    = argmax_j S[i,j] (top-1 retrieved caption)
#   • failure     = (pred_idx != paired_idx)
#
# Architecture preserved. New training regimes accessible via Cell 1.
# ============================================================================


# ---------------------------------------------------------------------------
# ABSTRACT BASE — shared MLP factory (matches repo ViLU semantics)
# ---------------------------------------------------------------------------

import math
from typing import List, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# BASE MLP CLASS
# ============================================================================

class AbstractViLU(nn.Module):
    def __init__(
        self,
        visual_dim: Optional[int] = None,
        textual_dim: Optional[int] = None,
        layers: Tuple[int, ...] = (512, 256, 128, 1),
        activation: str = "relu",
        negative_slope: float = 0.01,
        use_sigmoid: bool = False,
    ):
        super().__init__()
        self.visual_dim = int(visual_dim if visual_dim is not None else ACTIVE_FEATURE_DIM)
        self.textual_dim = int(textual_dim if textual_dim is not None else ACTIVE_TEXT_DIM)
        self.layers = list(layers)
        self.activation = str(activation)
        self.negative_slope = float(negative_slope)
        self.use_sigmoid = bool(use_sigmoid)

    def activation_layer(self) -> nn.Module:
        if self.activation == "relu":
            return nn.ReLU()
        if self.activation == "leaky_relu":
            return nn.LeakyReLU(self.negative_slope)
        raise ValueError(f"Unsupported activation: {self.activation}")

    def build_mlp(self, input_dim: int) -> nn.Sequential:
        modules: List[nn.Module] = [
            nn.Linear(input_dim, self.layers[0]),
            self.activation_layer(),
        ]
        for i in range(len(self.layers) - 1):
            modules.append(nn.Linear(self.layers[i], self.layers[i + 1]))
            if i < len(self.layers) - 2:
                modules.append(self.activation_layer())
        if self.use_sigmoid:
            modules.append(nn.Sigmoid())
        return nn.Sequential(*modules)


# ============================================================================
# ViLU FOR CAPTION RETRIEVAL (Paper-Faithful Architecture)
# ============================================================================

class ViLURetrievalAttention(AbstractViLU):
    """
    Standard ViLU for Caption Matching (Eqs. 4, 5, 6):
        1) Cross-attention:  z_t_alpha = Attention(Q=W_Q*z_v, K=W_K*Z_t, V=W_V*Z_t)
        2) Triplet Concat:   z_ViLU    = [z_v, z_t_hat, z_t_alpha]
        3) Failure Logits:   y_hat     = MLP(z_ViLU)
    """

    def __init__(
        self,
        visual_dim: Optional[int] = None,
        textual_dim: Optional[int] = None,
        proj_dim: Optional[int] = None,
        layers: Tuple[int, ...] = (512, 256, 128, 1),
        activation: str = "relu",
        use_sigmoid: bool = False,
        logit_scale: float = 100.0,
    ):
        super().__init__(
            visual_dim=visual_dim,
            textual_dim=textual_dim,
            layers=layers,
            activation=activation,
            use_sigmoid=use_sigmoid,
        )
        self.proj_dim = int(proj_dim if proj_dim is not None else self.visual_dim)
        self.logit_scale = float(logit_scale)

        # Cross-Attention Projections: W_Q, W_K, W_V (Eq. 5)
        self.W_Q = nn.Linear(self.visual_dim, self.proj_dim, bias=False)
        self.W_K = nn.Linear(self.textual_dim, self.proj_dim, bias=False)
        self.W_V = nn.Linear(self.textual_dim, self.proj_dim, bias=False)

        # Input to MLP: [z_v, z_t_hat, z_t_alpha]
        total_input_dim = self.visual_dim + self.textual_dim + self.proj_dim
        self.mlp = self.build_mlp(total_input_dim)

    def cross_attention(self, z_v: torch.Tensor, z_t_pool: torch.Tensor) -> torch.Tensor:
        """
        Computes image-specific contextual textual representation (Eq. 5):
            alpha = softmax( (W_Q * z_v) @ (W_K * z_t_pool).T / sqrt(d) )
            z_t_alpha = alpha @ (W_V * z_t_pool)
        """
        Q = self.W_Q(z_v)        # (B, proj_dim)
        K = self.W_K(z_t_pool)   # (K_candidates, proj_dim)
        V = self.W_V(z_t_pool)   # (K_candidates, proj_dim)

        scale = 1.0 / math.sqrt(self.proj_dim)
        attn_scores = (Q @ K.T) * scale            # (B, K_candidates)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        z_t_alpha = attn_weights @ V               # (B, proj_dim)
        return z_t_alpha

    def forward(
        self,
        v: torch.Tensor,
        t_pool: torch.Tensor,
        pred_labels: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            v: Visual embeddings (B, visual_dim)
            t_pool: Caption gallery embeddings (K_candidates, textual_dim)
            pred_labels: Optional precomputed top-1 indices (B,)
        Returns:
            logits: Predicted failure scores before or after sigmoid (B, 1)
        """
        # 1. Determine top-1 predicted caption index if not supplied
        if pred_labels is None:
            # Cosine similarity matching: S = v @ t_pool.T
            sims = self.logit_scale * (F.normalize(v, dim=-1) @ F.normalize(t_pool, dim=-1).T)
            pred_idx = torch.argmax(sims, dim=-1)
        else:
            pred_idx = pred_labels.long().to(v.device)

        z_t_hat = t_pool[pred_idx]                 # (B, textual_dim)

        # 2. Extract Cross-Modal Context (Eq. 4-5)
        z_t_alpha = self.cross_attention(v, t_pool)  # (B, proj_dim)

        # 3. Form ViLU Triplet Embedding (z_v, z_t_hat, z_t_alpha)
        z_ViLU = torch.cat([v, z_t_hat, z_t_alpha], dim=-1)

        # 4. Predict Failure Score via MLP (Eq. 6)
        return self.mlp(z_ViLU)


# ============================================================================
# BASELINE: LVU (Latent Visual Uncertainty)
# ============================================================================

class LVURetrieval(AbstractViLU):
    """Visual-only baseline head: g_theta(z_v)."""

    def __init__(
        self,
        visual_dim: Optional[int] = None,
        layers: Tuple[int, ...] = (512, 256, 128, 1),
        activation: str = "relu",
        use_sigmoid: bool = False,
    ):
        super().__init__(
            visual_dim=visual_dim,
            layers=layers,
            activation=activation,
            use_sigmoid=use_sigmoid,
        )
        self.mlp = self.build_mlp(self.visual_dim)

    def forward(
        self,
        v: torch.Tensor,
        t_pool: Optional[torch.Tensor] = None,
        pred_labels: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        return self.mlp(v)


# ============================================================================
# BUILDERS & LOSS FUNCTION
# ============================================================================

def build_vilu_model() -> nn.Module:
    """Canonical ViLU model."""
    return ViLURetrievalAttention(
        visual_dim=ACTIVE_FEATURE_DIM,
        textual_dim=ACTIVE_TEXT_DIM,
        proj_dim=ACTIVE_FEATURE_DIM,
        layers=(512, 256, 128, 1),
        activation="relu",
        use_sigmoid=False,   # Keep False when using BCEWithLogitsLoss
        logit_scale=CFG.logit_scale,
    )


def build_lvu_model() -> nn.Module:
    """Pure visual-only baseline."""
    return LVURetrieval(
        visual_dim=ACTIVE_FEATURE_DIM,
        layers=(512, 256, 128, 1),
        activation="relu",
        use_sigmoid=False,
    )


def weighted_bce_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Weighted BCE Loss as defined in Equations (9) and (10):
        w = log(1 + sum(1 - y) / (sum(y) + eps))
        Loss = -1/B * sum( w * y * log(sigma(logits)) + (1-y) * log(1 - sigma(logits)) )
    """
    targets = targets.float().view(-1, 1)
    num_errors = targets.sum()
    num_correct = (1.0 - targets).sum()

    # Eq. (10) Dynamic Batch Weighting
    weight = torch.log(1.0 + num_correct / (num_errors + 1e-6))

    loss_pos = weight * targets * F.logsigmoid(logits)
    loss_neg = (1.0 - targets) * F.logsigmoid(-logits)
    return -(loss_pos + loss_neg).mean()


# ---------------------------------------------------------------------------
# SANITY LOG
# ---------------------------------------------------------------------------
print(f"[ViLU/retrieval] Canonical architecture initialized.")
print(f"[ViLU/retrieval] Active dimensions: visual={ACTIVE_FEATURE_DIM}, textual={ACTIVE_TEXT_DIM}")
print(f"[ViLU/retrieval] Builders ready: build_vilu_model(), build_lvu_model()")

[ViLU/retrieval] Canonical architecture initialized.
[ViLU/retrieval] Active dimensions: visual=512, textual=512
[ViLU/retrieval] Builders ready: build_vilu_model(), build_lvu_model()


In [3]:
# CELL 3 — CLASS-LESS ATLAS (IMAGE ↔ CAPTION RETRIEVAL FAILURE DETECTION)
# ============================================================================
# ATLAS core logic:
#   • Identity-initialised sigmoid-gated visual/text projections
#   • Iteration-based projection warm-up freeze
#   • Cross-modal "gated cosine" fusion   pv = σ(<pv, pt>) · pv
#   • Six-dim CLIP-meta scalar features   [1−top1, top1−top2, entropy, top1, top2, cos]
#
# Adaptation for the class-less image-caption task:
#   • Text bank t = per-split CAPTION GALLERY T_pool ∈ R^{M × D}
#   • pred_labels = top-1 RETRIEVED CAPTION INDICES into T_pool
#   • Stats computed over the retrieval distribution
# ============================================================================


# ---------------------------------------------------------------------------
# PROB / PRED HELPER (retrieval-aware, class-less)
# ---------------------------------------------------------------------------

import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# ============================================================================
# PROB / PRED HELPER (retrieval-aware, class-less)
# ============================================================================

def _atlas_probs_and_preds(
    v: Tensor,
    t: Tensor,
    logit_scale: float = 100.0,
    pred_labels: Optional[Tensor] = None,
    target_logits: Optional[Tensor] = None,
    activation: str = "softmax",
    **kwargs,
) -> Tuple[Tensor, Tensor]:
    """
    Retrieval-probability + retrieved-caption-index resolver for ATLAS.

    Returns:
        probs : [B, M] softmax over the caption gallery
        preds : [B]    top-1 retrieved caption index in the gallery
    """
    if target_logits is None or str(activation).lower() == "sigmoid":
        # Cosine similarity matching over the caption gallery
        sims = logit_scale * (F.normalize(v, dim=-1) @ F.normalize(t, dim=-1).T)
        probs = torch.softmax(sims.float(), dim=-1)
    else:
        probs = target_scores_to_probs(
            torch.as_tensor(target_logits, device=v.device).float(),
            "softmax",
        ).to(v.device)

    if pred_labels is None:
        preds = probs.argmax(dim=-1)
    else:
        preds = torch.as_tensor(pred_labels, device=v.device, dtype=torch.long)

    return probs, preds


# ============================================================================
# CLIP-META SCALAR FEATURES (retrieval-aware)
# ============================================================================

def atlas_clip_meta(
    v: Tensor,
    t: Tensor,
    logit_scale: float = 100.0,
    pred_labels: Optional[Tensor] = None,
    target_logits: Optional[Tensor] = None,
    activation: str = "softmax",
    **kwargs,
) -> Tensor:
    """
    Six scalar CLIP-meta features per query, computed over the caption gallery:

        [ 1 − p_top1,                # retrieval uncertainty
          p_top1 − p_top2,           # retrieval margin
          entropy(p),                # distribution entropy
          p_top1,                    # confidence in retrieved caption
          p_top2,                    # runner-up mass
          <v, t[pred]>               # raw cosine to retrieved caption ]
    """
    probs, preds = _atlas_probs_and_preds(
        v,
        t,
        logit_scale=logit_scale,
        pred_labels=pred_labels,
        target_logits=target_logits,
        activation=activation,
    )

    k = min(2, probs.shape[1])
    top = probs.topk(k=k, dim=-1).values
    top1 = top[:, 0]
    top2 = top[:, 1] if k > 1 else torch.zeros_like(top1)
    entropy = -(probs * (probs + 1e-12).log()).sum(dim=-1)
    
    # Cosine similarity to the retrieved caption
    raw_cos = (F.normalize(v, dim=-1) * F.normalize(t[preds], dim=-1)).sum(dim=-1)

    return torch.stack([
        1.0 - top1,
        top1 - top2,
        entropy,
        top1,
        top2,
        raw_cos,
    ], dim=1).to(v.dtype)


# ============================================================================
# ATLAS — CANONICAL SCORER (No Cross-Attention)
# ============================================================================

class ATLASAttention(nn.Module):
    """
    ATLAS scorer over [projected image, projected retrieved-caption text, CLIP meta].

    Pipeline (retrieval-adapted):
        v          : [B, D]      visual query
        t          : [M, D]      per-split caption gallery
        pred       : [B]         top-1 retrieved caption idx in the gallery
        pv         : σ(<pv, pt>) · v_proj(v)         (gated visual)
        pt         : t_proj(t[pred])                 (projected retrieved caption)
        meta       : atlas_clip_meta(v, t, pred)     ([B, 6])
        logits     : scorer(cat([pv, pt, meta]))     ([B, 1])
    """

    def __init__(
        self,
        feat_dim: int = 512,
        n_iter_freeze_proj: int = 1000,
        dropout: float = 0.30,
        clip_meta_dim: int = 6,
    ):
        super().__init__()
        self.n_iter_freeze_proj = int(n_iter_freeze_proj)
        self._iter = 0
        self._proj_frozen = True
        self.clip_meta_dim = int(clip_meta_dim)
        self.logit_scale = CFG.logit_scale

        self.v_proj = nn.Linear(feat_dim, feat_dim, bias=False)
        self.t_proj = nn.Linear(feat_dim, feat_dim, bias=False)
        nn.init.eye_(self.v_proj.weight)
        nn.init.eye_(self.t_proj.weight)
        self._set_proj_frozen(True)

        self.scorer = nn.Sequential(
            nn.Linear(feat_dim * 2 + self.clip_meta_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        self.cap_proj = nn.Linear(feat_dim, feat_dim)

    def _set_proj_frozen(self, frozen: bool) -> None:
        for p in list(self.v_proj.parameters()) + list(self.t_proj.parameters()):
            p.requires_grad = not frozen
        self._proj_frozen = bool(frozen)

    def forward(
        self,
        v: Tensor,
        t: Tensor,
        pred_labels: Optional[Tensor] = None,
        true_labels: Optional[Tensor] = None,
        target_logits: Optional[Tensor] = None,
        activation: str = "softmax",
        **kwargs,
    ) -> Tuple[Tensor, Tensor, Tensor]:
        if self.training:
            self._iter += 1
            if self._proj_frozen and self._iter >= self.n_iter_freeze_proj:
                self._set_proj_frozen(False)

        probs, preds = _atlas_probs_and_preds(
            v,
            t,
            logit_scale=self.logit_scale,
            pred_labels=pred_labels,
            target_logits=target_logits,
            activation=activation,
        )
        txt = t[preds]

        pv = self.v_proj(v)
        pt = self.t_proj(txt)
        
        # Gated cosine fusion (Eq. pv = σ(<pv, pt>) * pv)
        pv = torch.sigmoid((pv * pt).sum(-1, keepdim=True)) * pv

        meta = atlas_clip_meta(
            v,
            t,
            logit_scale=self.logit_scale,
            pred_labels=preds,
            target_logits=target_logits,
            activation=activation,
        )

        logits = self.scorer(torch.cat([pv, pt, meta], dim=-1))

        v_hat = F.normalize(self.cap_proj(pv), dim=-1)
        t_hat = F.normalize(txt, dim=-1)
        return logits, v_hat, t_hat


# ============================================================================
# BUILDERS
# ============================================================================

def build_atlas_model() -> nn.Module:
    """
    Default ATLAS: original scorer over [pv, pt, clip_meta] with gated cosine
    fusion — no cross-attention.
    """
    return ATLASAttention(
        feat_dim=ACTIVE_FEATURE_DIM,
        n_iter_freeze_proj=CFG.n_iter_freeze_proj,
        dropout=0.30,
        clip_meta_dim=6,
    )


# ---------------------------------------------------------------------------
# SANITY LOG
# ---------------------------------------------------------------------------
print(f"[ATLAS/retrieval] Canonical architecture initialized (no cross-attention).")
print(f"[ATLAS/retrieval] Active dimensions: visual={ACTIVE_FEATURE_DIM}, textual={ACTIVE_TEXT_DIM}")
print(f"[ATLAS/retrieval] Builders ready: build_atlas_model()")

[ATLAS/retrieval] Canonical architecture initialized (no cross-attention).
[ATLAS/retrieval] Active dimensions: visual=512, textual=512
[ATLAS/retrieval] Builders ready: build_atlas_model()


In [4]:
# CELL 4 — CLASS-LESS SOARER (IMAGE ↔ CAPTION RETRIEVAL FAILURE DETECTION)
# ============================================================================

#
# Core architecture:
#   1. Learnable visual/text semantic projections: W_v(v), W_t(t) into R^{d_f}
#   2. Learnable temperature scaling: S = (W_v(v) @ W_t(t).T) / τ
#   3. Subspace posterior distribution: p = softmax(S, dim=-1)
#   4. Multi-order uncertainty representation (z_sem):
#        • First-order:  Top-m retrieval probabilities (p_top)
#        • Distributional: Subspace posterior entropy (ent)
#        • Boundary margin: Top-1 vs Top-2 retrieval gap (margin)
#        • Second-order interaction: Kronecker outer-product matrix (p_top ⊗ p_top)
#   5. Six-dimensional retrieval CLIP-meta scalar features
#   6. Deep Residual/LayerNorm classifier over [v, pred_txt, z_sem, clip_meta]
#
# Adaptation for class-less retrieval:
#   • t represents per-split CAPTION GALLERY T_pool ∈ R^{M × D}
#   • pred_labels are top-1 retrieved caption indices into T_pool
#   • Subspace similarity matrix S ∈ R^{B × M} evaluates queries vs all M captions
# ============================================================================


class SOARERModel(nn.Module):
    """
    SOARER adapted for image ↔ caption retrieval failure detection.

    Consumes:
        v            : [B, D_v] visual query batch
        t            : [M, D_t] per-split caption gallery embeddings
        pred_labels  : [B] top-1 retrieved caption indices into t
        true_labels  : [B] ground-truth paired caption indices
        target_logits: [B, M] optional precomputed logits from target model
    """
    def __init__(
        self,
        visual_dim: int = None,
        textual_dim: int = None,
        d_f: int = 256,
        m_top: int = 5,
        tau_init: float = 1.0,
        hidden_dim: int = 256,
        dropout: float = 0.30,
        clip_meta_dim: int = 6,
        logit_scale: float = 100.0,
    ):
        super().__init__()
        self.visual_dim = int(visual_dim if visual_dim is not None else ACTIVE_FEATURE_DIM)
        self.textual_dim = int(textual_dim if textual_dim is not None else ACTIVE_TEXT_DIM)
        self.d_f = int(d_f)
        self.m_top = int(m_top)
        self.clip_meta_dim = int(clip_meta_dim)
        self.logit_scale = float(logit_scale)

        # 1. Learned subspace projection matrices
        self.W_v = nn.Linear(self.visual_dim, self.d_f, bias=False)
        self.W_t = nn.Linear(self.textual_dim, self.d_f, bias=False)
        nn.init.xavier_uniform_(self.W_v.weight)
        nn.init.xavier_uniform_(self.W_t.weight)

        # 2. Learnable temperature scaling
        self.log_tau = nn.Parameter(
            torch.tensor(np.log(max(tau_init, 1e-3)), dtype=torch.float32)
        )

        # 3. Second-order semantic representation dimensionality:
        #    p_top (m) + ent (1) + margin (1) + kron (m * m)
        d_sem = (self.m_top * self.m_top) + self.m_top + 2
        input_dim = self.visual_dim + self.textual_dim + d_sem + self.clip_meta_dim

        # 4. Multi-layer failure prediction head
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

        # Auxiliary caption projection head for cross-modal alignment regularization
        self.cap_proj = nn.Linear(self.d_f, self.textual_dim)

    def forward(
        self,
        v: Tensor,
        t: Tensor,
        pred_labels: Optional[Tensor] = None,
        true_labels: Optional[Tensor] = None,
        target_logits: Optional[Tensor] = None,
        activation: str = "softmax",
        **kwargs,
    ) -> Tuple[Tensor, Tensor, Tensor]:
        B = v.shape[0]
        M = t.shape[0]

        # 1. Resolve top-1 predicted/retrieved caption index
        if pred_labels is None:
            if target_logits is not None and str(activation).lower() != "sigmoid":
                probs_target = target_scores_to_probs(
                    torch.as_tensor(target_logits, device=v.device).float(), "softmax"
                )
                preds = probs_target.argmax(dim=-1).long()
            else:
                sims_direct = self.logit_scale * (v @ t.T)
                preds = sims_direct.argmax(dim=-1).long()
        else:
            preds = torch.as_tensor(pred_labels, device=v.device, dtype=torch.long)

        pred_txt = t[preds]  # [B, D_t]

        # 2. Compute learned subspace similarities over the full caption gallery
        v_proj = self.W_v(v)      # [B, d_f]
        t_proj = self.W_t(t)      # [M, d_f]
        tau = self.log_tau.exp().clamp(min=1e-3, max=100.0)

        S = (v_proj @ t_proj.T) / tau
        p = torch.softmax(S, dim=-1)

        # 3. Extract Multi-Order Semantic Features (z_sem)
        m = min(self.m_top, M)
        p_top = p.topk(m, dim=-1, largest=True, sorted=True).values  # [B, m]
        if m < self.m_top:
            p_top = F.pad(p_top, (0, self.m_top - m))

        ent = -(p * (p + 1e-12).log()).sum(dim=-1, keepdim=True)    # [B, 1]
        margin = (p_top[:, :1] - p_top[:, 1:2]) if self.m_top > 1 else torch.zeros_like(ent)
        kron = (p_top.unsqueeze(2) * p_top.unsqueeze(1)).reshape(B, -1)  # [B, m*m]

        z_sem = torch.cat([p_top, ent, margin, kron], dim=-1)       # [B, d_sem]

        # 4. Extract retrieval CLIP-meta scalar features
        meta = atlas_clip_meta(
            v,
            t,
            logit_scale=self.logit_scale,
            pred_labels=preds,
            true_labels=true_labels,
            target_logits=target_logits,
            activation=activation,
        )  # [B, 6]

        # 5. Concatenate all evidence paths and predict failure logits
        combined_features = torch.cat([v, pred_txt, z_sem, meta], dim=-1)
        logits = self.classifier(combined_features)  # [B, 1]

        # 6. Normalized representations for auxiliary alignment regularizer
        v_hat = F.normalize(self.cap_proj(v_proj), dim=-1)
        t_hat = F.normalize(pred_txt, dim=-1)

        return logits, v_hat, t_hat


# ---------------------------------------------------------------------------
# BUILDER
# ---------------------------------------------------------------------------

def build_soarer_model() -> nn.Module:
    """
    Builds the default SOARER failure detection model configured with active feature dims.
    """
    return SOARERModel(
        visual_dim=ACTIVE_FEATURE_DIM,
        textual_dim=ACTIVE_TEXT_DIM,
        d_f=256,
        m_top=5,
        tau_init=1.0,
        hidden_dim=256,
        dropout=0.30,
        clip_meta_dim=6,
        logit_scale=CFG.logit_scale,
    )


# ---------------------------------------------------------------------------
# SANITY LOG
# ---------------------------------------------------------------------------

print(f"[SOARER/retrieval] active dims: visual={ACTIVE_FEATURE_DIM}, textual={ACTIVE_TEXT_DIM}")
print(f"[SOARER/retrieval] subspace_dim(d_f)=256, m_top=5, clip_meta_dim=6, logit_scale={CFG.logit_scale}")
print(f"[SOARER/retrieval] builder registered: build_soarer_model")

[SOARER/retrieval] active dims: visual=512, textual=512
[SOARER/retrieval] subspace_dim(d_f)=256, m_top=5, clip_meta_dim=6, logit_scale=100.0
[SOARER/retrieval] builder registered: build_soarer_model


In [ ]:
# CELL 5 — MULTI-TARGET FAILURE DETECTION PIPELINE (EXECUTION CELL)
# ============================================================================
# Task: Image ↔ Text Retrieval Failure Detection (Multi-Target Zero-Shot)
# Baselines: MCM, LVU, ViLU (using ViLUWeightedBCELoss)
# Proposed: SOARER, ATLAS
# Ensemble: Bayesian Fusion (over SOARER + ATLAS)
# Spearman rho: Computed between SOARER & ATLAS on test split
#
# NOTE: The train/val/test split ratio is configured in CFG.split_ratio
#        (defined in CELL 1). Edit it there to change the partition.
# ============================================================================

import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from pathlib import Path
from scipy.stats import spearmanr
from torch.utils.data import DataLoader
from torchvision import transforms

try:
    from skopt import gp_minimize
    from skopt.space import Real
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False

# ============================================================================
# 1. PIPELINE CONFIGURATION & SEEDS
# ============================================================================
SEEDS = CFG.seeds

print(f"{'#'*78}")
print(f"# Target VLMs: {CFG.zero_shot_target_models}")
print(f"# Split ratio: {CFG.split_ratio}  |  Tag: {CFG.split_tag}")
print(f"# Baselines: MCM, LVU, ViLU  |  Proposed: SOARER, ATLAS  |  Ensemble: Bayesian Fusion")
print(f"{'#'*78}\n")

# Safe Loss Fallbacks
DefaultBCELoss = globals().get("OrdinaryBCELoss", globals().get("FailureBCELoss", nn.BCEWithLogitsLoss))

# ============================================================================
# 2. DATASET PAIRS PROVIDER RESOLUTION
# ============================================================================
pairs_provider = None
candidate_names = ["dataset_pairs_provider", "load_dataset_pairs", 
                   "get_dataset_pairs", "get_pairs", "all_pairs"]

for name in candidate_names:
    if name in globals():
        pairs_provider = globals()[name]
        print(f"[Dataset] Using resolved provider: '{name}'")
        break

if pairs_provider is None:
    for k, v in list(globals().items()):
        if not k.startswith("_") and callable(v) and any(sub in k.lower() for sub in ["dataset", "load_data", "get_data", "pairs"]):
            pairs_provider = v
            print(f"[Dataset] Using resolved custom loader: '{k}'")
            break

# ============================================================================
# 3. DATASET BUNDLE INITIALIZATION
# ============================================================================
bundle = build_dataset_bundle(CFG, pairs_provider=pairs_provider)

# ============================================================================
# 4. DYNAMIC TARGET-SPECIFIC FEATURE EXTRACTION HELPER
# ============================================================================
class SafeRGBConverter:
    def __call__(self, img):
        return img.convert("RGB") if isinstance(img, Image.Image) else img


def _extract_tensor(out):
    """Safely unpacks raw PyTorch tensors from HuggingFace/OpenCLIP output objects."""
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "image_embeds") and out.image_embeds is not None:
        return out.image_embeds
    if hasattr(out, "text_embeds") and out.text_embeds is not None:
        return out.text_embeds
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        lhs = out.last_hidden_state
        return lhs[:, 0] if lhs.ndim == 3 else lhs
    if isinstance(out, (tuple, list)):
        return out[0]
    return torch.as_tensor(out)


def encode_dataset_for_target(bundle, target_name):
    print(f"\n[Features] Encoding visual and textual embeddings for target: {target_name}...")
    clean_device_cache()
    
    t_name = str(target_name).upper().strip()
    
    if "SIGLIP" in t_name:
        loaded = False
        siglip_hf_name = getattr(CFG, "siglip_model", "google/siglip-base-patch16-224")
        
        try:
            from transformers import AutoProcessor, AutoModel
            processor = AutoProcessor.from_pretrained(siglip_hf_name)
            model = AutoModel.from_pretrained(siglip_hf_name).to(DEVICE)
            model.eval()
            
            target_transform = transforms.Compose([
                transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
                SafeRGBConverter(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
            
            def get_image_feats(imgs):
                out = model.get_image_features(pixel_values=imgs.to(DEVICE))
                return _extract_tensor(out)
                
            def get_text_feats(texts):
                inputs = processor(text=list(texts), padding="max_length", max_length=64, truncation=True, return_tensors="pt")
                attn = inputs["attention_mask"].to(DEVICE) if "attention_mask" in inputs else None
                out = model.get_text_features(input_ids=inputs["input_ids"].to(DEVICE), attention_mask=attn)
                return _extract_tensor(out)
                
            loaded = True
            print("   Loaded SigLIP via HuggingFace transformers.")
        except Exception as e:
            print(f"   HuggingFace SigLIP failed ({e}). Falling back to OpenCLIP SigLIP...")

        if not loaded:
            import open_clip
            model, _, target_transform = open_clip.create_model_and_transforms('ViT-B-16-SigLIP', pretrained='webli', device=DEVICE)
            tokenizer = open_clip.get_tokenizer('ViT-B-16-SigLIP')
            model.eval()
            
            def get_image_feats(imgs):
                return _extract_tensor(model.encode_image(imgs.to(DEVICE)))
                
            def get_text_feats(texts):
                tokens = tokenizer(list(texts)).to(DEVICE)
                return _extract_tensor(model.encode_text(tokens))
            print("   Loaded SigLIP via OpenCLIP (ViT-B-16-SigLIP / webli).")

    elif "OPENCLIP" in t_name:
        import open_clip
        arch = getattr(CFG, "openclip_model", "ViT-B-32")
        pt = getattr(CFG, "openclip_pretrained", "laion2b_s34b_b79k")
        model, _, target_transform = open_clip.create_model_and_transforms(arch, pretrained=pt, device=DEVICE)
        tokenizer = open_clip.get_tokenizer(arch)
        model.eval()
        
        def get_image_feats(imgs):
            return _extract_tensor(model.encode_image(imgs.to(DEVICE)))
            
        def get_text_feats(texts):
            tokens = tokenizer(list(texts)).to(DEVICE)
            return _extract_tensor(model.encode_text(tokens))
        print(f"   Loaded OpenCLIP ({arch} / {pt}).")

    else:  # CLIP (Default)
        import clip
        clip_arch = getattr(CFG, "clip_model", "ViT-B/32")
        model, target_transform = clip.load(clip_arch, device=DEVICE)
        model.eval()
        
        def get_image_feats(imgs):
            return _extract_tensor(model.encode_image(imgs.to(DEVICE)))
            
        def get_text_feats(texts):
            tokens = clip.tokenize(list(texts), truncate=True).to(DEVICE)
            return _extract_tensor(model.encode_text(tokens))
        print(f"   Loaded OpenAI CLIP ({clip_arch}).")

    for split in ["train", "val", "test"]:
        if hasattr(bundle[split], "transform"):
            bundle[split].transform = target_transform
        if hasattr(bundle[split], "base") and hasattr(bundle[split].base, "transform"):
            bundle[split].base.transform = target_transform

    def extract_split(ds):
        loader = DataLoader(
            ds, batch_size=getattr(CFG, "feature_batch_size", 256),
            shuffle=False, num_workers=getattr(CFG, "num_workers", 2)
        )
        all_v, all_t = [], []
        with torch.no_grad():
            for batch in loader:
                imgs = batch["image"].to(DEVICE)
                caption_key = next((k for k in ["caption", "text", "tweet_text", "sentence"] if k in batch), None)
                if caption_key is not None:
                    texts = batch[caption_key]
                elif "target" in batch and "class_names" in bundle:
                    class_names = bundle["class_names"]
                    texts = [CFG.prompt_template.format(class_names[int(idx)]) for idx in batch["target"]]
                else:
                    raise KeyError("Could not resolve caption/text key in dataset batch.")
                    
                v_feats = _extract_tensor(get_image_feats(imgs))
                t_feats = _extract_tensor(get_text_feats(texts))
                
                v_feats = v_feats / v_feats.norm(dim=-1, keepdim=True)
                t_feats = t_feats / t_feats.norm(dim=-1, keepdim=True)
                
                all_v.append(v_feats.float().cpu())
                all_t.append(t_feats.float().cpu())
                
        return torch.cat(all_v), torch.cat(all_t)

    tr_v, tr_t = extract_split(bundle["train"])
    va_v, va_t = extract_split(bundle["val"])
    te_v, te_t = extract_split(bundle["test"])

    encoded = {
        "train": {"v": tr_v, "t": tr_t},
        "val": {"v": va_v, "t": va_t},
        "test": {"v": te_v, "t": te_t}
    }
    
    del model
    clean_device_cache()
    return encoded

# ============================================================================
# 5. MODEL CONFIG REGISTRIES
# ============================================================================
def build_baseline_configs():
    configs = {}
    if "LVUNet" in globals() or "build_lvu_model" in globals():
        configs["LVU"] = {
            "builder": globals().get("build_lvu_model", lambda: LVUNet(ACTIVE_FEATURE_DIM, ACTIVE_TEXT_DIM)),
            "loss_builder": lambda: DefaultBCELoss(),
            "balanced_sampling": False,
        }
    if "ViLURetrievalAttention" in globals() or "build_vilu_model" in globals():
        configs["ViLU"] = {
            "builder": globals().get("build_vilu_model", lambda: ViLURetrievalAttention(
                visual_dim=ACTIVE_FEATURE_DIM, textual_dim=ACTIVE_TEXT_DIM, proj_dim=ACTIVE_FEATURE_DIM
            )),
            "loss_builder": lambda: ViLUWeightedBCELoss(),
            "balanced_sampling": False,
        }
    return configs


def build_proposed_configs():
    configs = {}
    if "SOARER" in globals() or "build_soarer_model" in globals():
        configs["SOARER"] = {
            "builder": globals().get("build_soarer_model", lambda: SOARER(
                visual_dim=ACTIVE_FEATURE_DIM, textual_dim=ACTIVE_TEXT_DIM, proj_dim=256
            )),
            "regime": "R5_balanced_BCE",
            "balanced_sampling": True,
            "loss_builder": lambda: DefaultBCELoss(),
        }
    if "ATLASAttention" in globals() or "build_atlas_model" in globals():
        configs["ATLAS"] = {
            "builder": globals().get("build_atlas_model", lambda: ATLASAttention(
                visual_dim=ACTIVE_FEATURE_DIM, textual_dim=ACTIVE_TEXT_DIM, proj_dim=256
            )),
            "regime": "R6_balanced_BCE_rank_random_0.1",
            "balanced_sampling": True,
            "loss_builder": lambda: DefaultBCELoss(),
            "ranking_type": "random",
            "ranking_weight": 0.1,
        }
    return configs

# ============================================================================
# 6. BAYESIAN FUSION OPTIMIZER
# ============================================================================
def sigmoid_norm(scores):
    s = np.asarray(scores, dtype=float)
    return 1.0 / (1.0 + np.exp(-s))


def fit_bayes_weights(val_matrix, val_correct, seed=42):
    n_models = val_matrix.shape[1]
    
    def objective(w):
        w = np.array(w)
        w = w / (w.sum() + 1e-12)
        ensemble_scores = val_matrix @ w
        try:
            fpr = get_fpr95(ensemble_scores[val_correct], ensemble_scores[~val_correct])
            auc = get_auroc(ensemble_scores[val_correct], ensemble_scores[~val_correct])
            return float(fpr - auc)
        except Exception:
            return 0.0

    if SKOPT_AVAILABLE:
        bounds = [Real(0.0, 1.0) for _ in range(n_models)]
        res = gp_minimize(
            objective, bounds, n_calls=80, n_initial_points=20,
            random_state=seed, verbose=False
        )
        best_w = np.array(res.x)
    else:
        rng = np.random.default_rng(seed)
        best_score, best_w = float("inf"), np.ones(n_models) / n_models
        for _ in range(10000):
            w = rng.dirichlet(np.ones(n_models))
            score = objective(w)
            if score < best_score:
                best_score = score
                best_w = w
                
    return best_w / (best_w.sum() + 1e-12)

# ============================================================================
# 7. PIPELINE LOOP OVER TARGET VLMs
# ============================================================================
target_models = CFG.zero_shot_target_models
raw_results = []
hp_lock = {}

for target_name in target_models:
    print(f"\n{'='*78}")
    print(f" TARGET VLM: {target_name}")
    print(f"{'='*78}")
    
    target_encoded = encode_dataset_for_target(bundle, target_name)
    
    tr_v, tr_t = target_encoded["train"]["v"], target_encoded["train"]["t"]
    va_v, va_t = target_encoded["val"]["v"], target_encoded["val"]["t"]
    te_v, te_t = target_encoded["test"]["v"], target_encoded["test"]["t"]
    
    set_active_dims(target_encoded, tr_t)
    
    tr_y = torch.arange(len(tr_v))
    va_y = torch.arange(len(va_v))
    te_y = torch.arange(len(te_v))
    
    with torch.no_grad():
        tr_sim = CFG.logit_scale * tr_v @ tr_t.T
        tr_preds = tr_sim.argmax(dim=-1)
        tr_correct = (tr_y == tr_preds.cpu()).numpy()
        tr_failures = (~tr_correct).astype(float)
        
        va_sim = CFG.logit_scale * va_v @ va_t.T
        va_preds = va_sim.argmax(dim=-1)
        va_correct = (va_y == va_preds.cpu()).numpy()
        va_failures = (~va_correct).astype(float)
        
        te_sim = CFG.logit_scale * te_v @ te_t.T
        te_preds = te_sim.argmax(dim=-1)
        te_correct = (te_y == te_preds.cpu()).numpy()
        te_failures = (~te_correct).astype(float)

    with torch.no_grad():
        te_sims = CFG.logit_scale * te_v @ te_t.T
        te_mcm = -te_sims.max(dim=-1)[0].cpu().numpy()
        
    mcm_eval = evaluate_failure_scores(
        scores=te_mcm, labels=te_y, image_features=te_v,
        text_features=te_t, logit_scale=CFG.logit_scale,
        correct_mask=te_correct, target_name=target_name
    )
    print(f"[Zero-shot | {target_name} | Target Retrieval Test Accuracy: {100*mcm_eval['accuracy']:.2f}%]")
    print(f"[Zero-shot | {target_name} | MCM (Test)] AUC: {mcm_eval['auc']:.4f} | FPR95: {mcm_eval['fpr95']:.4f} | Cohen's d: {mcm_eval['cohen_d']:.4f}")

    for seed in SEEDS:
        raw_results.append({
            "target": target_name, "seed": seed, "model": "MCM",
            "accuracy": mcm_eval["accuracy"], "auc": mcm_eval["auc"], 
            "fpr95": mcm_eval["fpr95"], "cohen_d": mcm_eval["cohen_d"]
        })

    baseline_configs = build_baseline_configs()
    proposed_configs = build_proposed_configs()
    
    for seed in SEEDS:
        print(f"\n--- Running Seed {seed} ---")
        seed_predictions_val = {}
        seed_predictions_test = {}
        is_anchor = (seed == SEEDS[0])
        
        all_models = {**baseline_configs, **proposed_configs}
        for model_name, m_cfg in all_models.items():
            print(f" Training: {model_name}...")
            
            fixed_pw = m_cfg.get("fixed_pos_weight", None)
            bal_samp = m_cfg.get("balanced_sampling", False)
            rank_type = m_cfg.get("ranking_type", "none")
            rank_weight = m_cfg.get("ranking_weight", 0.0)
            
            hp_key = f"{target_name}_{model_name}"
            fixed_selection = hp_lock.get(hp_key) if not is_anchor else None
            
            model, train_info = grid_search_then_train_final(
                model_builder=m_cfg["builder"],
                train_v=tr_v, train_y=tr_y,
                val_v=va_v, val_y=va_y,
                text_features=tr_t, seed=seed,
                fixed_selection=fixed_selection,
                loss_builder=m_cfg.get("loss_builder"),
                train_failure_labels=tr_failures,
                train_pred_labels=tr_preds,
                val_pred_labels=va_preds,
                val_correct_mask=va_correct,
                target_name=target_name,
                fixed_pos_weight=fixed_pw,
                balanced_sampling=bal_samp,
                ranking_weight=rank_weight,
                ranking_type=rank_type
            )
            
            if is_anchor:
                hp_lock[hp_key] = train_info["selected"]
                
            val_scores = score_torch_model(model, va_v, va_t, pred_labels=va_preds)
            test_scores = score_torch_model(model, te_v, te_t, pred_labels=te_preds)
            
            seed_predictions_val[model_name] = val_scores
            seed_predictions_test[model_name] = test_scores
            
            eval_res = evaluate_failure_scores(
                scores=test_scores, labels=te_y, image_features=te_v,
                text_features=te_t, logit_scale=CFG.logit_scale,
                correct_mask=te_correct, target_name=target_name
            )
            
            print(f"   [Seed {seed} | {target_name} | {model_name:<7}] AUC: {eval_res['auc']:.4f} | FPR95: {eval_res['fpr95']:.4f} | Cohen's d: {eval_res['cohen_d']:.4f}")
            
            raw_results.append({
                "target": target_name, "seed": seed, "model": model_name,
                "accuracy": eval_res["accuracy"], "auc": eval_res["auc"], 
                "fpr95": eval_res["fpr95"], "cohen_d": eval_res["cohen_d"]
            })
            clean_device_cache()
            
        # ------------------------------------------------------------------------
        # 8. BAYESIAN FUSION ENSEMBLE (Over SOARER & ATLAS)
        # ------------------------------------------------------------------------
        ensemble_members = ["SOARER", "ATLAS"]
        valid_members = [m for m in ensemble_members if m in seed_predictions_val and m in seed_predictions_test]
        
        if len(valid_members) >= 2:
            val_matrix = np.column_stack([sigmoid_norm(seed_predictions_val[m]) for m in valid_members])
            test_matrix = np.column_stack([sigmoid_norm(seed_predictions_test[m]) for m in valid_members])
            
            ensemble_weights = fit_bayes_weights(val_matrix, va_correct, seed=seed)
            fused_test = test_matrix @ ensemble_weights
            fuse_eval = evaluate_failure_scores(
                scores=fused_test, labels=te_y, image_features=te_v,
                text_features=te_t, logit_scale=CFG.logit_scale,
                correct_mask=te_correct, target_name=target_name
            )
            
            weights_str = ", ".join([f"{m}={w:.3f}" for m, w in zip(valid_members, ensemble_weights)])
            print(f"   [Seed {seed} | {target_name} | BayesFusion] AUC: {fuse_eval['auc']:.4f} | FPR95: {fuse_eval['fpr95']:.4f} | Cohen's d: {fuse_eval['cohen_d']:.4f} ({weights_str})")
            
            raw_results.append({
                "target": target_name, "seed": seed, "model": "BayesFusion",
                "accuracy": fuse_eval["accuracy"], "auc": fuse_eval["auc"], 
                "fpr95": fuse_eval["fpr95"], "cohen_d": fuse_eval["cohen_d"]
            })
            
            # ------------------------------------------------------------------------
            # 9. SPEARMAN RANK CORRELATION (SOARER vs ATLAS)
            # ------------------------------------------------------------------------
            s_scores = seed_predictions_test["SOARER"]
            a_scores = seed_predictions_test["ATLAS"]
            rho, _ = spearmanr(s_scores, a_scores)
            
            print(f"   [Seed {seed} | {target_name} | Spearman (SOARER vs ATLAS)] rho = {rho:.4f}")
            
            raw_results.append({
                "target": target_name, "seed": seed, "model": "Spearman_SOARER_ATLAS",
                "accuracy": float("nan"), "auc": float("nan"), "fpr95": float("nan"), 
                "cohen_d": float("nan"), "spearman_rho": float(rho)
            })

# ============================================================================
# 10. OUTPUT DATA TABLES & PERSISTENCE
# ============================================================================
df_raw = pd.DataFrame(raw_results)
df_raw.to_csv(RESULTS_DIR / f"{CFG.dataset_name}_multitarget_final_results_per_run.csv", index=False)

print("\n" + "="*78)
print(" PER-SEED RAW RESULTS BREAKDOWN (ALL SEEDS & TARGETS)")
print("="*78)
display(df_raw)

summary_rows = []
spearman_rows = []

for (target, model), grp in df_raw.groupby(["target", "model"], sort=False):
    if model == "Spearman_SOARER_ATLAS":
        rhos = grp["spearman_rho"].dropna().to_numpy()
        if len(rhos) > 0:
            spearman_rows.append({
                "Target": target,
                "Spearman_Rho_Mean": float(np.mean(rhos)),
                "Spearman_Rho_SD": float(np.std(rhos, ddof=1)) if len(rhos) > 1 else 0.0
            })
        continue
        
    row = {"target": target, "model": model, "n_runs": len(grp)}
    for metric in ["accuracy", "auc", "fpr95", "cohen_d"]:
        vals = grp[metric].dropna().to_numpy()
        row[f"{metric}_mean"] = float(np.mean(vals)) if len(vals) > 0 else float("nan")
        row[f"{metric}_sd"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(RESULTS_DIR / f"{CFG.dataset_name}_multitarget_final_summary.csv", index=False)

df_spearman = pd.DataFrame(spearman_rows)

print("\n" + "="*78)
print(f" AGGREGATE SUMMARY TABLES (Mean ± Std over {len(SEEDS)} seeds)")
print("="*78)
for metric in ["accuracy", "auc", "fpr95", "cohen_d"]:
    table_data = []
    models_listed = [m for m in df_summary["model"].unique()]
    
    for target in target_models:
        t_row = {"Target": target}
        for m in models_listed:
            sub = df_summary[(df_summary["target"] == target) & (df_summary["model"] == m)]
            if not sub.empty:
                m_val = sub[f"{metric}_mean"].values[0]
                s_val = sub[f"{metric}_sd"].values[0]
                t_row[m] = f"{m_val:.4f} ± {s_val:.4f}"
            else:
                t_row[m] = "N/A"
        table_data.append(t_row)
        
    df_table = pd.DataFrame(table_data)
    df_table.to_csv(RESULTS_DIR / f"{CFG.dataset_name}_{metric}_table.csv", index=False)
    print(f"\n[{metric.upper()} Table (Mean ± Std)]")
    display(df_table)

print("\n[Spearman Rank Correlation (SOARER vs ATLAS)]")
df_spearman.to_csv(RESULTS_DIR / f"{CFG.dataset_name}_SOARER_ATLAS_spearman_rho_table.csv", index=False)
display(df_spearman)

final_payload = {
    "dataset": CFG.dataset_name,
    "split_tag": CFG.split_tag,
    "split_ratio": list(CFG.split_ratio),
    "seeds": list(SEEDS),
    "hyperparameters": hp_lock,
    "metrics_summary": df_summary.to_dict(orient="records"),
    "spearman_correlation": df_spearman.to_dict(orient="records")
}

with open(RESULTS_DIR / f"{CFG.dataset_name}_multitarget_final_results.json", "w") as f:
    json.dump(final_payload, f, indent=4)

print(f"\nExecution finished successfully. All results saved to: '{RESULTS_DIR}'")

##############################################################################
# Target VLMs: ('CLIP', 'SigLIP', 'OpenCLIP')
# Split ratio: (0.925, 0.05, 0.025)  |  Tag: split_92.5_5_2.5
# Baselines: MCM, LVU, ViLU  |  Proposed: SOARER, ATLAS  |  Ensemble: Bayesian Fusion
##############################################################################

[Dataset] Using resolved provider: 'dataset_pairs_provider'
[CrisisMMD] Split sizes [92.5/5/2.5]: train=16726, val=904, test=452

 TARGET VLM: CLIP

[Features] Encoding visual and textual embeddings for target: CLIP...
   Loaded OpenAI CLIP (ViT-B/32).
